# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis and time window

My lane is content refresh opportunity scoring.

One row represents one pseudonymized content item for one client on one report date in the fact_content_daily_performance table.

For development, I will use the mid-panel month March 2026, covering 2026-03-01 through 2026-03-31. I will not use the _sample table for label or feature development because it contains only June 2026, the final month of the panel, which must remain a sealed test period.

Client history begins on different dates, so I will verify the relevant search and analytics availability flags before using each row. This is necessary because unavailable GA4 values may be stored as zeros rather than true observations.

In [ ]:
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.sql(grain_query).df()

print("Duplicate grain combinations found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature, label, context, and excluded

**Prediction goal:**  
I will score and rank content items by their risk of a future decline in search impressions. The result will support a content team in deciding which pages should be reviewed for a possible refresh.

**Features:**  
The final feature frame will contain no more than five features, calculated only from information available before the prediction moment:

1. `past_impressions` — total GSC impressions in the feature window.
2. `past_clicks` — total GSC clicks in the feature window.
3. `past_ctr` — past clicks divided by past impressions.
4. `past_avg_position` — the content item's search position during the feature window.
5. `gsc_observed_days` — the number of days with usable GSC data.

**Label / proxy:**  
The proxy label will indicate whether the content item's average daily search impressions decline in the later outcome window compared with the earlier feature window. This is a proxy for refresh need; it does not prove that refreshing the content will cause recovery.

**Context fields:**  
`client_hash_id`, `content_hash_id`, and `report_date` will be used for grouping, joining, filtering, and time-based splitting. They will not be used as model features.

**Availability filter:**  
`gsc_data_available` will be used as a filter and must be checked with `IS TRUE`. It is not a predictive feature.

**Deliberately excluded:**  
I will exclude GA4 fields from this first feature frame because GA4 is unavailable for some clients and unavailable values may be zero-filled. I will also exclude identifiers and all future-window or label-derived columns from the honest model because they would cause leakage.

**Output:**  
The analysis will produce a ranked list of pseudonymized content items for human review, with higher scores representing a greater observed risk of future search-performance decline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
slice_query = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet('{march_path}')
"""

slice_summary = con.sql(slice_query).df()
slice_summary

,row_count,client_count,content_count,earliest_date,latest_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [ ]:
availability_query = f"""
WITH all_rows AS (
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{march_path}')
),
available_rows AS (
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
)
SELECT
    all_rows.total_rows,
    available_rows.available_rows,
    ROUND(
        100.0 * available_rows.available_rows / all_rows.total_rows,
        2
    ) AS available_pct
FROM all_rows
CROSS JOIN available_rows
"""

availability_summary = con.sql(availability_query).df()
availability_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,available_pct
0,9841378,3611061,36.69


### Five-feature frame

I use March 1–21, 2026 as the feature window and March 22–31, 2026 as the later outcome window.

The proxy label `future_decline` equals `1` when the content item's average daily impressions in the outcome window are at least 20% lower than in the feature window. It equals `0` otherwise.

The five model features are:

1. `past_impressions` — knowable at the decision moment because it uses only impressions observed during March 1–21.
2. `past_clicks` — knowable at the decision moment because it uses only clicks observed during March 1–21.
3. `past_ctr` — knowable at the decision moment because it is calculated only from past clicks and past impressions.
4. `past_avg_position` — knowable at the decision moment because it uses only GSC position information from March 1–21.
5. `gsc_observed_days` — knowable at the decision moment because it counts only usable GSC days already observed in the feature window.

`client_hash_id` and `content_hash_id` remain context fields and are not model features.

In [ ]:
feature_query = f"""
WITH item_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                     AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                     AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                     AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    past_impressions,
    past_clicks,

    ROUND(
        100.0 * past_clicks / NULLIF(past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * past_sum_position / NULLIF(past_impressions, 0),
        4
    ) AS past_avg_position,

    gsc_observed_days,

    CASE
        WHEN
            (1.0 * outcome_impressions / outcome_observed_days)
            <
            0.80 * (1.0 * past_impressions / gsc_observed_days)
        THEN 1
        ELSE 0
    END AS future_decline

FROM item_windows
WHERE
    gsc_observed_days >= 7
    AND outcome_observed_days >= 5
    AND past_impressions >= 100
"""

feature_frame = con.sql(feature_query).df()

print("Rows in feature frame:", len(feature_frame))
print("Label distribution:")
print(feature_frame["future_decline"].value_counts(dropna=False))

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in feature frame: 85990
Label distribution:
future_decline
0    58052
1    27938
Name: count, dtype: int64


,client_hash_id,content_hash_id,past_impressions,past_clicks,past_ctr,past_avg_position,gsc_observed_days,future_decline
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,645.0,2.0,0.3101,4.5752,21,0
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,109.0,0.0,0.0000,3.6514,20,1
2,client_73cda7b4e4f265ea,content_05434271b257bb68,894.0,3.0,0.3356,5.5817,21,0
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2229.0,16.0,0.7178,3.6402,21,1
4,client_73cda7b4e4f265ea,content_2662845f598544ef,122.0,1.0,0.8197,8.3852,21,1


### Deliberate leakage experiment

First, I evaluate a quick model using only the five honest features that are available before the prediction moment.

I then deliberately add `label_leak`, which is a direct copy of `future_decline`. This column contains the answer the model is supposed to predict, so it would never be available at the real decision moment. A near-perfect score after adding it is evidence of target leakage, not genuine model quality.

After demonstrating the score jump, I remove `label_leak` and retain the honest score.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = [
    "past_impressions",
    "past_clicks",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
]

X_honest = feature_frame[honest_features].astype(float)
y = feature_frame["future_decline"].astype(int)
groups = feature_frame["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X_honest, y, groups=groups)
)

def calculate_auc(X):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    )

    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    probabilities = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    return roc_auc_score(
        y.iloc[test_idx],
        probabilities,
    )

# Honest model
honest_auc = calculate_auc(X_honest)

# Deliberately leaked model
X_leaked = X_honest.copy()
X_leaked["label_leak"] = y

leaked_auc = calculate_auc(X_leaked)

# Remove the leaking column
X_final = X_leaked.drop(columns=["label_leak"])

honest_auc_after_removal = calculate_auc(X_final)

print(f"Honest ROC-AUC: {honest_auc:.4f}")
print(f"Leaked ROC-AUC: {leaked_auc:.4f}")
print(
    "Honest ROC-AUC after removing leakage:",
    f"{honest_auc_after_removal:.4f}",
)
print(
    "Leak column still present:",
    "label_leak" in X_final.columns,
)

Honest ROC-AUC: 0.5537
Leaked ROC-AUC: 1.0000
Honest ROC-AUC after removing leakage: 0.5537
Leak column still present: False


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation: coverage bias

This feature frame includes only content items with at least seven usable GSC days in the feature window, at least five usable days in the outcome window, and at least 100 past impressions.

Therefore, the results do not represent content with sparse search history or very low visibility. The proxy also measures an observed decline, but it cannot prove that refreshing a page would cause its performance to recover.

In [ ]:
limitation_check = (
    feature_frame[
        [
            "past_impressions",
            "gsc_observed_days",
        ]
    ]
    .agg(["min", "median", "max"])
    .round(2)
)

limitation_check

,past_impressions,gsc_observed_days
min,100.0,7.0
median,680.0,21.0
max,273012.0,21.0


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.